In [ ]:
import os
import json
import gzip
import requests
import pandas as pd

In [ ]:
# 1- Downloading the station data.
!curl "https://bulk.meteostat.net/v2/stations/full.json.gz" --output "full.json.gz"

In [ ]:
# 2- Opeining the station data.
with gzip.open('/content/full.json.gz', 'rt') as f:
  data = json.load(f)

# 3- Filtring the station data to US only.
filtered_data = []
for item in data:
  if item['country'] == 'US':
    filtered_data.append(item)

# 4- Filtring the station data based on recording time. 2014-2024
# Define the date range
start_date = pd.to_datetime('2014-01-01')
end_date = pd.to_datetime('2024-01-01')

station_ids = []

for station in filtered_data:
    # Get hourly and daily start dates, handling missing data
    hourly_start = station['inventory'].get('hourly', {}).get('start')
    daily_start = station['inventory'].get('daily', {}).get('start')

    # Convert to datetime objects, handling None values
    hourly_start = pd.to_datetime(hourly_start) if hourly_start else None
    daily_start = pd.to_datetime(daily_start) if daily_start else None

    # Check if either hourly or daily start dates are within the range
    if (hourly_start is not None and hourly_start >= start_date and hourly_start <= end_date) or \
       (daily_start is not None and daily_start >= start_date and daily_start <= end_date):
        station_ids.append(station['id'])


In [ ]:
# 4- Downloading the weather data per each station
folder_name = 'daily_data'
os.makedirs(folder_name, exist_ok=True)

# Iterate through station IDs and download data
for station_id in station_ids:
    url = f"https://bulk.meteostat.net/v2/daily/{station_id}.csv.gz"
    file_path = os.path.join(folder_name, f"{station_id}.csv.gz")

    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad responses

        with open(file_path, 'wb') as f:
            f.write(response.content)

        print(f"Downloaded data for station {station_id}")

    except requests.exceptions.RequestException as e:
        print(f"Error downloading data for station {station_id}: {e}")

In [ ]:
# 5- Processing the data and saving the .csv format.

csv_folder = 'daily_csv_data_processed'
os.makedirs(csv_folder, exist_ok=True)

column_names = ['date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']

# Create a dictionary to map station IDs to their information
station_info = {station['id']: station for station in filtered_data}

# Iterate through files in the hourly_data folder
for filename in os.listdir('daily_data'):
    if filename.endswith('.csv.gz'):
        file_path = os.path.join('daily_data', filename)
        csv_file_path = os.path.join(csv_folder, filename[:-3])  # Remove .gz extension

        try:
            with gzip.open(file_path, 'rt', encoding='utf-8') as f:
                df = pd.read_csv(f, header=None, names=column_names)

            # Add a 'station' column with the station ID
            station_id = filename[:-7]  # Extract station ID from filename
            df['station'] = station_id

            # Add columns for name, country, region, etc.
            info = station_info.get(station_id, {})  # Get station information
            df['name'] = info.get('name', {}).get('en', '')  # Extract English name
            df['country'] = info.get('country', '')
            df['region'] = info.get('region', '')
            df['latitude'] = info.get('location', {}).get('latitude', '')
            df['longitude'] = info.get('location', {}).get('longitude', '')
            df['elevation'] = info.get('location', {}).get('elevation', '')
            df['timezone'] = info.get('timezone', '')
            df.to_csv(csv_file_path, index=False)
            print(f"Processed and saved {filename} to {csv_file_path}")
            gc.collect()
        except Exception as e:
            print(f"Error processing {filename}: {e}")

In [ ]:
# Colab Link: https://colab.research.google.com/drive/14TSKlMpXm6PD2E0Ycph-P_zR_x4jCXh-?usp=sharing